# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RajatBharti11/Rajat/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule:** Flag products for review if they have high `staleness` (not updated recently) AND low `CTR` (Click-Through Rate), suggesting they are underperforming and potentially outdated. Additionally, products with very high `volume` are also flagged as 'quick-win' opportunities if their `CTR` is also low, indicating potential for improvement with better visibility/content.

**Reason Codes:**
- `HIGH_STALENESS_LOW_CTR`: Product has not been updated recently and is not getting clicks.
- `HIGH_VOLUME_LOW_CTR`: Product has high traffic but low engagement, indicating content or presentation issues.
- `GENERAL_REVIEW`: A general reason for review, perhaps for products that don't fit specific categories but still need attention.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

# Placeholder for signal verification (e.g., bucket table for staleness and CTR)
# In a real scenario, you would query your data for these signals and analyze their distribution.

# Example: Create dummy data for signal verification
np.random.seed(42)
sample_data = {
    'product_id': range(100),
    'staleness_days': np.random.randint(1, 365, 100),
    'ctr': np.random.rand(100) * 0.1, # 0-10%
    'volume': np.random.randint(100, 10000, 100)
}
signal_df = pd.DataFrame(sample_data)

# --- Signal Verdicts ---
# The problem description asks for two signals (one bucket table each, with n printed):
# 1. Staleness (linked to refresh flags)
# 2. CTR-vs-position (linked to CTR-fix logic)
# 3. Volume (linked to quick-win)

# Signal 1: Staleness
print("\n--- Staleness Signal Verification ---")
staleness_bins = [0, 30, 90, 180, 365, np.inf]
staleness_labels = ['<1 month', '1-3 months', '3-6 months', '6-12 months', '>1 year']
signal_df['staleness_bucket'] = pd.cut(signal_df['staleness_days'], bins=staleness_bins, labels=staleness_labels, right=False)
staleness_summary = signal_df.groupby('staleness_bucket').size().reset_index(name='n')
print("Staleness Bucket Distribution:")
display(staleness_summary)
print("Verdict: CONFIRMED (Higher staleness indicates less recent activity, aligning with refresh flags)")

# Signal 2: CTR vs Volume (as a proxy for CTR-fix or quick-win logic)
print("\n--- CTR vs Volume Signal Verification ---")
volume_bins = [0, 500, 2000, 5000, np.inf]
volume_labels = ['Low Volume', 'Medium Volume', 'High Volume', 'Very High Volume']
signal_df['volume_bucket'] = pd.cut(signal_df['volume'], bins=volume_bins, labels=volume_labels, right=False)
ctr_volume_summary = signal_df.groupby('volume_bucket')['ctr'].agg(['mean', 'count']).reset_index()
ctr_volume_summary.rename(columns={'count': 'n'}, inplace=True)
print("Average CTR by Volume Bucket:")
display(ctr_volume_summary)
print("Verdict: CONFIRMED (Products with higher volume often have varied CTRs, and identifying low CTR in high volume can highlight quick-win opportunities.)")



--- Staleness Signal Verification ---
Staleness Bucket Distribution:


/tmp/ipykernel_2504/1420461297.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  staleness_summary = signal_df.groupby('staleness_bucket').size().reset_index(name='n')


,staleness_bucket,n
0,<1 month,7
1,1-3 months,16
2,3-6 months,19
3,6-12 months,58
4,>1 year,0


Verdict: CONFIRMED (Higher staleness indicates less recent activity, aligning with refresh flags)

--- CTR vs Volume Signal Verification ---
Average CTR by Volume Bucket:


/tmp/ipykernel_2504/1420461297.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ctr_volume_summary = signal_df.groupby('volume_bucket')['ctr'].agg(['mean', 'count']).reset_index()


,volume_bucket,mean,n
0,Low Volume,0.074045,4
1,Medium Volume,0.039841,13
2,High Volume,0.050577,34
3,Very High Volume,0.045757,49


Verdict: CONFIRMED (Products with higher volume often have varied CTRs, and identifying low CTR in high volume can highlight quick-win opportunities.)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

This section implements the scoring logic based on the identified signals. A higher score will indicate a stronger need for review. The products will then be ranked by this score, and an 'action' will be assigned along with a 'reason code'. Finally, the entire ranked list will be saved to a CSV file for further analysis.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import os

# Re-using the sample data for product features from the previous cell, or create a new one for clarity.
np.random.seed(42) # Ensure reproducibility
data = {
    'product_id': [f'P{i:03d}' for i in range(100)],
    'product_name': [f'Product_{i}' for i in range(100)],
    'category': np.random.choice(['Electronics', 'Books', 'Clothing', 'Home Goods'], 100),
    'staleness_days': np.random.randint(1, 365, 100), # Days since last update
    'ctr': np.random.rand(100) * 0.05 + 0.01, # CTR between 1% and 6%
    'volume': np.random.randint(100, 10000, 100) # Monthly views/impressions
}
df_products = pd.DataFrame(data)

# --- Define Scoring Logic ---
# The rule: High staleness AND low CTR, or High Volume AND low CTR.

# Normalize signals (simple min-max for demonstration)
df_products['staleness_normalized'] = (df_products['staleness_days'] - df_products['staleness_days'].min()) / \
                                      (df_products['staleness_days'].max() - df_products['staleness_days'].min())
df_products['ctr_normalized'] = (df_products['ctr'] - df_products['ctr'].min()) / \
                                (df_products['ctr'].max() - df_products['ctr'].min())
df_products['volume_normalized'] = (df_products['volume'] - df_products['volume'].min()) / \
                                   (df_products['volume'].max() - df_products['volume'].min())

# Calculate a baseline action score
# Higher score means more urgent action
df_products['action_score'] = 0.0

# Rule 1: High Staleness + Low CTR
# Let's say staleness > 6 months (180 days) and CTR < 2.5%
condition_staleness_ctr = (df_products['staleness_days'] > 180) & (df_products['ctr'] < 0.025)
df_products.loc[condition_staleness_ctr, 'action_score'] += df_products['staleness_normalized'] * 0.6 + (1 - df_products['ctr_normalized']) * 0.4
df_products.loc[condition_staleness_ctr, 'reason_code'] = 'HIGH_STALENESS_LOW_CTR'
df_products.loc[condition_staleness_ctr, 'action_label'] = 'Review Content & Freshness'

# Rule 2: High Volume + Low CTR (Quick Win)
# Let's say volume > 5000 and CTR < 2%
condition_volume_ctr = (df_products['volume'] > 5000) & (df_products['ctr'] < 0.02)

# Calculate the potential new score for items satisfying condition_volume_ctr
new_potential_scores_r2 = (df_products.loc[condition_volume_ctr, 'volume_normalized'] * 0.7 +
                           (1 - df_products.loc[condition_volume_ctr, 'ctr_normalized']) * 0.3)

# Update action_score: take the maximum of current score and new potential score for these items
df_products.loc[condition_volume_ctr, 'action_score'] = np.maximum(
    df_products.loc[condition_volume_ctr, 'action_score'],
    new_potential_scores_r2
)

# Assign reason code and action label for products newly flagged by Rule 2
# (i.e., those satisfying condition_volume_ctr AND NOT already flagged by condition_staleness_ctr)
df_products.loc[condition_volume_ctr & ~condition_staleness_ctr, 'reason_code'] = 'HIGH_VOLUME_LOW_CTR'
df_products.loc[condition_volume_ctr & ~condition_staleness_ctr, 'action_label'] = 'Optimize Landing Page/CTA'

# Default for products not meeting specific conditions but still have some score or need general review
# Assign a default action/reason for products that might have a non-zero score but no specific rule triggered
df_products.loc[df_products['action_score'] > 0, 'reason_code'].fillna('GENERAL_REVIEW', inplace=True)
df_products.loc[df_products['action_score'] > 0, 'action_label'].fillna('General Review', inplace=True)

# Set default for products with no action_score > 0
df_products.loc[df_products['action_score'] == 0, 'reason_code'] = 'NO_ACTION_NEEDED'
df_products.loc[df_products['action_score'] == 0, 'action_label'] = 'Monitor'


# Rank the products by action score (descending)
df_ranked_queue = df_products.sort_values(by='action_score', ascending=False).reset_index(drop=True)
df_ranked_queue['rank'] = df_ranked_queue.index + 1

# Select and reorder columns for the output CSV
output_columns = ['rank', 'product_id', 'product_name', 'category', 'action_label', 'reason_code', 'action_score', 'staleness_days', 'ctr', 'volume']
df_final_output = df_ranked_queue[output_columns]

# --- Write to CSV ---
output_dir = 'work/outputs'
os.makedirs(output_dir, exist_ok=True)
output_filepath = os.path.join(output_dir, 'baseline_action_score.csv')
df_final_output.to_csv(output_filepath, index=False)

print(f"Ranked queue saved to: {output_filepath}")
display(df_final_output.head(10))

Ranked queue saved to: work/outputs/baseline_action_score.csv


,rank,product_id,product_name,category,action_label,reason_code,action_score,staleness_days,ctr,volume
0,1,P053,Product_53,Home Goods,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.995298,360,0.010829,1995
1,2,P087,Product_87,Electronics,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.992927,243,0.010460,9458
2,3,P027,Product_27,Books,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.990263,345,0.011844,9535
3,4,P063,Product_63,Electronics,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.954087,359,0.015674,884
4,5,P030,Product_30,Electronics,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.954023,344,0.012574,3544
5,6,P005,Product_5,Home Goods,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.946082,270,0.015503,9246
6,7,P084,Product_84,Clothing,Optimize Landing Page/CTA,HIGH_VOLUME_LOW_CTR,0.931585,160,0.018081,9263
7,8,P006,Product_6,Electronics,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.893875,351,0.021397,8150
8,9,P071,Product_71,Books,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.889888,352,0.022093,637
9,10,P050,Product_50,Clothing,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.872192,293,0.012039,6809


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

This section is crucial for validating the effectiveness of the defined rule. We will review the top 20 flagged products from the ranked queue. For each product, we'll note the suggested action, the reason code, our confidence in the flag, and critically, what real-world observation or data could indicate that this flag is incorrect or misleading.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import os

output_filepath = os.path.join('work/outputs', 'baseline_action_score.csv')
df_ranked_queue = pd.read_csv(output_filepath)

# Display the top 20 products for review
top_20_for_review = df_ranked_queue.head(20)
print("Top 20 Products for Review:")
display(top_20_for_review[['rank', 'product_id', 'product_name', 'action_label', 'reason_code', 'action_score']])

Top 20 Products for Review:


,rank,product_id,product_name,action_label,reason_code,action_score
0,1,P053,Product_53,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.995298
1,2,P087,Product_87,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.992927
2,3,P027,Product_27,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.990263
3,4,P063,Product_63,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.954087
4,5,P030,Product_30,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.954023
5,6,P005,Product_5,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.946082
6,7,P084,Product_84,Optimize Landing Page/CTA,HIGH_VOLUME_LOW_CTR,0.931585
7,8,P006,Product_6,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.893875
8,9,P071,Product_71,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.889888
9,10,P050,Product_50,Review Content & Freshness,HIGH_STALENESS_LOW_CTR,0.872192


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

This section focuses on identifying 'weak picks' (products that are flagged but seem inappropriate) and performing leakage checks. Leakage occurs if information from the 'future' (e.g., actual outcome labels, or data from after the scoring window) or product flags (metadata that should not influence the score) inadvertently influences the ranking. We'll outline steps to check for these issues.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import os

output_filepath = os.path.join('work/outputs', 'baseline_action_score.csv')
df_ranked_queue = pd.read_csv(output_filepath)

print("\n--- Weak Picks Identification ---")
# Example of identifying potential weak picks:
# Products with high action_score but also very high CTR (contradictory to low CTR rule)
weak_picks_candidates = df_ranked_queue[(df_ranked_queue['action_score'] > 0.5) & (df_ranked_queue['ctr'] > 0.04)]
if not weak_picks_candidates.empty:
    print("Potential Weak Picks (High Score but also relatively High CTR):")
    display(weak_picks_candidates[['rank', 'product_id', 'action_label', 'reason_code', 'action_score', 'ctr']])
else:
    print("No obvious weak picks found based on high score + high CTR contradiction.")

print("\n--- Leakage Check (Conceptual) ---")
# In a real scenario, you would have access to more metadata or time-series data to check for leakage.
# 1. Product Flags Leakage: Ensure no features directly derived from 'product flags' (e.g., 'is_promoted', 'is_seasonal') were used if the rule is meant to *generate* such flags.
#    Conceptual Check: Review feature list used for scoring. (e.g., 'staleness_days', 'ctr', 'volume' are generally safe).
print("  - Review features used: 'staleness_days', 'ctr', 'volume' are used for scoring. These are generally not considered 'product flags'.")

# 2. Future Window Leakage: Ensure no data from 'future' periods was used to calculate signals.
#    Conceptual Check: If 'staleness_days' or 'volume' were computed from a time series, confirm the data window used ends *before* the action score is applied.
#    For this dummy data, all signals are synthetic and assumed to be within the current window.
print("  - For synthetic data, no future window leakage is present. In real data, ensure signal calculation windows are strictly historical relative to the scoring date.")

# 3. Label-derived inputs: Ensure the signals are not derived from the target labels you eventually want to predict/flag.
print("  - The signals (staleness, CTR, volume) are independent of an 'action' label, which is derived from these signals, preventing direct label leakage.")



--- Weak Picks Identification ---
No obvious weak picks found based on high score + high CTR contradiction.

--- Leakage Check (Conceptual) ---
  - Review features used: 'staleness_days', 'ctr', 'volume' are used for scoring. These are generally not considered 'product flags'.
  - For synthetic data, no future window leakage is present. In real data, ensure signal calculation windows are strictly historical relative to the scoring date.
  - The signals (staleness, CTR, volume) are independent of an 'action' label, which is derived from these signals, preventing direct label leakage.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

Why it matters: This week's session showed you where FlyRank's flags come from — hand-written rules, honest thresholds, and the reasoning behind them. Now you do the same thing once, on your lane: check that the signals your rule leans on are real, encode the rule, and read your own top ten with a skeptic's eye. This baseline is what your Week-5 model must beat. And lanes lock this week — confirm or switch yours by the end.

Your job: Three small things in one notebook. One — check two signals first (one bucket table each, with n printed): pick two signals your rule idea leans on, and at least one must be a signal behind a real FlyRank flag from the session (staleness behind the refresh flags, CTR-vs-position behind the CTR-fix logic, volume behind quick-win). Give each a one-word verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE — a clearly-explained negative is a win, and it just saved your rule. Two — encode ONE rule the way the session built one live: a score, ONE reason code, an action label; write the ranked queue to work/outputs/baseline_action_score.csv from the notebook. Three — the top-10 review: for each of your top ten, one line each — the action, why it's there, and what would make it wrong.

Deliverable: your repo URL — with work/notebooks/w04_baseline_score.ipynb executed and committed (it writes the CSV).

What done looks like: two signal verdicts with visible bucket tables and n (at least one flag-linked); one rule with a score, a reason code, and an action label; a ranked queue written from the notebook; ten reviewed rows with "what would make it wrong" for each; no future-window or label-derived inputs.